In [1]:
"""
GraphFrames Demo
==========================

A tiny but complete example that shows how to:

* create a graph from Spark DataFrames,
* run basic analytics (degrees, PageRank, connected components …),
* query the graph with motifs and BFS,
* persist results for later use.

The data model is an airline network:
    - vertices = airports (identified by IATA code)
    - edges    = direct flights (directed, one‑way)

Run it with:
    $ spark-submit --packages graphframes:graphframes:0.8 graphframes_demo.py (extract to a file first)
or inside a notebook after installing the `graphframes` Python wrapper.
"""

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from graphframes import GraphFrame

In [2]:
def print_section(title: str) -> None:
    """
    Utility to print a clear visual separator.
    Handy when you run the script interactively.
    """
    line = "=" * 40
    print(f"\n{line}\n{title}\n{line}")

In [3]:
# Choose the package that matches your Spark version
graphframes_pkg = "io.graphframes:graphframes-spark3_2.12:0.10.0"  # Spark 3.x

# Create a SparkSession with GraphFrames JAR automatically adde
spark = (
    SparkSession.builder.master("local[*]")
    .appName("GraphFrames Demo")
    .config("spark.executor.instances", "2")
    .config("spark.driver.memory", "2g")
    .config("spark.jars.packages", graphframes_pkg)
    .config("spark.checkpoint.dir", ".")
    .getOrCreate()
)
# Silence some verbose logs – makes classroom output cleaner
spark.sparkContext.setLogLevel("ERROR")

25/11/24 11:21:57 WARN Utils: Your hostname, Tonys-MacBook-Air-2.local resolves to a loopback address: 127.0.0.1; using 172.20.10.3 instead (on interface en0)
25/11/24 11:21:57 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Ivy Default Cache set to: /Users/tonytziorvas/.ivy2/cache
The jars for the packages stored in: /Users/tonytziorvas/.ivy2/jars
io.graphframes#graphframes-spark3_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-f99788f8-de49-44a0-9478-11f9e22a6cdb;1.0
	confs: [default]
	found io.graphframes#graphframes-spark3_2.12;0.10.0 in central
	found io.graphframes#graphframes-graphx-spark3_2.12;0.10.0 in central
:: resolution report :: resolve 53ms :: artifacts dl 3ms
	:: modules in use:
	io.graphframes#graphframes-graphx-spark3_2.12;0.10.0 from central in [default]
	io.graphframes#graphframes-spark3_2.12;0.10.0 from central in [default]
	---------------------------------------------------------------------
	|   

:: loading settings :: url = jar:file:/Users/tonytziorvas/Documents/1.Personal/Lectures/BSc/TMD146/Code/.venv/lib/python3.11/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


25/11/24 11:21:57 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


In [4]:
# Build the vertices (airports) and edges (flights) DataFrames
print_section("🔹 Creating vertices (airports)")

vertices = spark.createDataFrame(
    [
        ("ATL", "Atlanta Hartsfield‑Jackson"),
        ("ORD", "Chicago O'Hare"),
        ("LAX", "Los Angeles International"),
        ("DFW", "Dallas/Fort Worth"),
        ("JFK", "New York John F. Kennedy"),
        ("SFO", "San Francisco International"),
    ],
    ["id", "name"],  # column names required by GraphFrames
)
vertices.show(truncate=False)

print_section("🔹 Creating edges (directed flights)")

edges = spark.createDataFrame(
    [
        # src  dst   (optional) weight – here we just use 1 for each flight
        ("ATL", "ORD"),
        ("ORD", "LAX"),
        ("LAX", "DFW"),
        ("DFW", "ATL"),
        ("ATL", "LAX"),
        ("JFK", "ATL"),
        ("JFK", "SFO"),
        ("SFO", "ORD"),
        ("SFO", "LAX"),
    ],
    ["src", "dst"],
)
edges.show(truncate=False)
# Construct the GraphFrame object

print_section("🔹 Instantiating GraphFrame")
g = GraphFrame(vertices, edges)


🔹 Creating vertices (airports)


+---+---------------------------+
|id |name                       |
+---+---------------------------+
|ATL|Atlanta Hartsfield‑Jackson |
|ORD|Chicago O'Hare             |
|LAX|Los Angeles International  |
|DFW|Dallas/Fort Worth          |
|JFK|New York John F. Kennedy   |
|SFO|San Francisco International|
+---+---------------------------+


🔹 Creating edges (directed flights)
+---+---+
|src|dst|
+---+---+
|ATL|ORD|
|ORD|LAX|
|LAX|DFW|
|DFW|ATL|
|ATL|LAX|
|JFK|ATL|
|JFK|SFO|
|SFO|ORD|
|SFO|LAX|
+---+---+


🔹 Instantiating GraphFrame


/Users/tonytziorvas/Documents/1.Personal/Lectures/BSc/TMD146/Code/.venv/lib/python3.11/site-packages/pyspark/sql/dataframe.py:168: UserWarning: DataFrame.sql_ctx is an internal property, and will be removed in future releases. Use DataFrame.sparkSession instead.
  warnings.warn(


### Simple graph metrics


In [5]:
# 5a – Degrees (in, out, total)
print_section("📈 Vertex degree statistics")
deg = g.degrees  # total degree (in + out)
deg.show()

# In‑degree and out‑degree are also available directly:
in_deg = g.inDegrees
out_deg = g.outDegrees
print_section("🔽 In‑degrees (incoming flights)")
in_deg.show()
print_section("🔼 Out‑degrees (outgoing flights)")
out_deg.show()


📈 Vertex degree statistics


/Users/tonytziorvas/Documents/1.Personal/Lectures/BSc/TMD146/Code/.venv/lib/python3.11/site-packages/pyspark/sql/dataframe.py:147: UserWarning: DataFrame constructor is internal. Do not directly use it.
  warnings.warn("DataFrame constructor is internal. Do not directly use it.")


+---+------+
| id|degree|
+---+------+
|ATL|     4|
|ORD|     3|
|LAX|     4|
|DFW|     2|
|JFK|     2|
|SFO|     3|
+---+------+


🔽 In‑degrees (incoming flights)
+---+--------+
| id|inDegree|
+---+--------+
|ORD|       2|
|LAX|       3|
|DFW|       1|
|ATL|       2|
|SFO|       1|
+---+--------+


🔼 Out‑degrees (outgoing flights)
+---+---------+
| id|outDegree|
+---+---------+
|ATL|        2|
|ORD|        1|
|LAX|        1|
|DFW|        1|
|JFK|        2|
|SFO|        2|
+---+---------+



### PageRank (global importance)


In [6]:
print_section("🌐 PageRank (10 iterations, resetProbability=0.15)")
pr = g.pageRank(resetProbability=0.15, maxIter=10)
pr.vertices.select("id", "pagerank").orderBy(F.col("pagerank").desc()).show(
    truncate=False
)


🌐 PageRank (10 iterations, resetProbability=0.15)
+---+-------------------+
|id |pagerank           |
+---+-------------------+
|LAX|1.6382146537334226 |
|DFW|1.5843182665958129 |
|ATL|1.5395026211452392 |
|ORD|0.8742144585255248 |
|SFO|0.21375000000000008|
|JFK|0.15000000000000005|
+---+-------------------+



### Connected components (weakly‑connected)


In [7]:
print_section("🔗 Connected Components")
cc = g.connectedComponents()
cc.select("id", "component").orderBy("component").show(truncate=False)


🔗 Connected Components
+---+-------------+
|id |component    |
+---+-------------+
|DFW|1056561954816|
|JFK|1056561954816|
|ATL|1056561954816|
|ORD|1056561954816|
|LAX|1056561954816|
|SFO|1056561954816|
+---+-------------+



### Triangle count (how many 3‑node cycles each vertex belongs to)


In [8]:
print_section("🔺 Triangle Count")
tc = g.triangleCount()
tc.select("id", "count").orderBy(F.col("count").desc()).show(truncate=False)


🔺 Triangle Count
+---+-----+
|id |count|
+---+-----+
|LAX|3    |
|ATL|2    |
|ORD|2    |
|DFW|1    |
|SFO|1    |
|JFK|0    |
+---+-----+



### Motif Finding – pattern matching in the graph

In [ ]:
# Example: Find all “two‑hop” flight itineraries A → B → C
print_section("🔎 Motif: two‑hop itineraries (A → B → C)")
motifs = g.find("(a)-[e1]->(b); (b)-[e2]->(c)")

# Keep only distinct airport codes, drop the edge columns for brevity
motifs.select("a.id", "b.id", "c.id").distinct().orderBy("a.id", "b.id", "c.id").show(
    truncate=False
)


🔎 Motif: two‑hop itineraries (A → B → C)
+---+---+---+
|id |id |id |
+---+---+---+
|ATL|LAX|DFW|
|ATL|ORD|LAX|
|DFW|ATL|LAX|
|DFW|ATL|ORD|
|JFK|ATL|LAX|
|JFK|ATL|ORD|
|JFK|SFO|LAX|
|JFK|SFO|ORD|
|LAX|DFW|ATL|
|ORD|LAX|DFW|
|SFO|LAX|DFW|
|SFO|ORD|LAX|
+---+---+---+



In [10]:
# 7️⃣  Breadth‑First Search (BFS) – shortest‑hop route between two airports

print_section("🧭 BFS: shortest route from JFK to DFW")
bfs = g.bfs(
    fromExpr="id = 'JFK'",
    toExpr="id = 'DFW'",
    maxPathLength=4,  # limit search depth for speed in a demo
)
bfs.show(truncate=False)


🧭 BFS: shortest route from JFK to DFW
+-------------------------------+----------+----------------------------------+----------+--------------------------------+----------+------------------------+
|from                           |e0        |v1                                |e1        |v2                              |e2        |to                      |
+-------------------------------+----------+----------------------------------+----------+--------------------------------+----------+------------------------+
|{JFK, New York John F. Kennedy}|{JFK, SFO}|{SFO, San Francisco International}|{SFO, LAX}|{LAX, Los Angeles International}|{LAX, DFW}|{DFW, Dallas/Fort Worth}|
|{JFK, New York John F. Kennedy}|{JFK, ATL}|{ATL, Atlanta Hartsfield‑Jackson} |{ATL, LAX}|{LAX, Los Angeles International}|{LAX, DFW}|{DFW, Dallas/Fort Worth}|
+-------------------------------+----------+----------------------------------+----------+--------------------------------+----------+------------------------+



### Persist results (optional)

In [11]:
# Here we write the PageRank results to a local folder.
output_dir = "data"
print_section(f"💾 Saving PageRank results to {output_dir}/pagerank")

# force a single CSV file for simplicity
pr.vertices.select("id", "pagerank").coalesce(1).write.mode("overwrite").option(
    "header", "true"
).csv(f"{output_dir}/pagerank")


💾 Saving PageRank results to data/pagerank


In [12]:
# Clean up

print_section("✅ Demo finished – stopping Spark")
spark.stop()


✅ Demo finished – stopping Spark
